In [1]:
import bw2data, bw2io
import bw2calc
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import os
import sys

In [2]:
sys.path.append('/Users/susierwu/dpLCA_main/') 
from utils import *
from utils.newbw2method_dpLCIA_addBW25 import * 

In [3]:
bw2data.projects.set_current('ei311')

In [4]:
mybio = bw2data.Database("ecoinvent-3.11-biosphere")
len(mybio)

9795

### read in pre-calculated .nc LCIA dataset, d-IRF for a new IRF metric calculation

In [7]:
cf_point = xr.open_dataset('../../../dpLCIA/AGWPCO2_fixed_approach/output_dpGWP_fixedCO2_modC/CF_GWP1_100_perSSP_MY_majorghgs_ModC.nc')
cf_point

<xarray.Dataset> Size: 44kB
Dimensions:    (SSP: 6, ModelYear: 3, Year: 100)
Coordinates:
  * SSP        (SSP) object 48B '119' '126' '245' '434' '460' '585'
  * ModelYear  (ModelYear) int32 12B 2030 2040 2050
  * Year       (Year) int32 400B 1 2 3 4 5 6 7 8 9 ... 93 94 95 96 97 98 99 100
Data variables:
    CO2_GWP    (SSP, ModelYear, Year) float64 14kB ...
    CH4_GWP    (SSP, ModelYear, Year) float64 14kB ...
    N2O_GWP    (SSP, ModelYear, Year) float64 14kB ...

### read in premise_GWP, all minor_ghg using static amount 

In [8]:
prem_gwp_dfraw = pd.read_excel("../../../dpLCIA/LCIA/premise_gwp/lcia_gwp2021_100a_w_bio.xlsx")
prem_gwp_dfraw.head()

,name,categories,amount
0,Bromopropane,air::unspecified,0.052
1,Butane,air::urban air close to ground,0.006
2,Butane,air::non-urban air or from high stacks,0.006
3,Butane,"air::low population density, long-term",0.006
4,Butane,air::lower stratosphere + upper troposphere,0.006


### calling the assign_dpGWP class, see what it looks like for final CF

In [9]:
xx = assign_dpGWP(premise_gwp100_inputdf = prem_gwp_dfraw, cf_inputds = cf_point, ssp = '119', fairMY = 2030 )
minorg, allg = xx.get_minorand_allGHG()
cc = xx.prep_empty_C(allg)
#cc.head()
pcc = xx.assign_minorghg_to_C_GWP100(cc, minorg)
fcc = xx.assign_majorghg_dCC(pcc)
fcc

,Bromopropane,Butane,"Carbon monoxide, fossil","Carbon monoxide, from soil or biomass stock","Carbon monoxide, non-fossil",Chloroform,Ethane,"Ethane, 1,1,1,2-tetrafluoro-, HFC-134a","Ethane, 1,1,1-trichloro-, HCFC-140","Ethane, 1,1,1-trifluoro-, HFC-143a",...,"Carbon dioxide, non-fossil","Carbon dioxide, in air","Carbon dioxide, to soil or biomass stock","Carbon dioxide, from soil or biomass stock","Carbon dioxide, fossil","Carbon dioxide, non-fossil, resource correction","Methane, from soil or biomass stock","Methane, fossil","Methane, non-fossil",Dinitrogen monoxide
GWP1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.941822,-0.941822,-0.941822,0.941822,0.941822,-0.941822,129.474722,129.474722,129.474722,170.824170
GWP2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.939335,-0.939335,-0.939335,0.939335,0.939335,-0.939335,128.415469,128.415469,128.415469,175.169989
GWP3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.940314,-0.940314,-0.940314,0.940314,0.940314,-0.940314,127.033721,127.033721,127.033721,179.096258
GWP4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.941997,-0.941997,-0.941997,0.941997,0.941997,-0.941997,125.360343,125.360343,125.360343,182.631564
GWP5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.945282,-0.945282,-0.945282,0.945282,0.945282,-0.945282,123.486220,123.486220,123.486220,185.805955
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GWP96,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.019608,-1.019608,-1.019608,1.019608,1.019608,-1.019608,30.341629,30.341629,30.341629,184.333983
GWP97,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.019692,-1.019692,-1.019692,1.019692,1.019692,-1.019692,30.096084,30.096084,30.096084,183.820148
GWP98,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.019770,-1.019770,-1.019770,1.019770,1.019770,-1.019770,29.855031,29.855031,29.855031,183.305769
GWP99,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.019843,-1.019843,-1.019843,1.019843,1.019843,-1.019843,29.618343,29.618343,29.618343,182.790926


### run MY2030/2040&2050 and all SSP together, for GWP we only have pGWP100 now, using static GWP100 for minorGHGs

In [10]:
for mmy in [2030, 2040, 2050]: 
    for sp in ['119', '126', '245', '434', '460', '585']: 
        xx = assign_dpGWP(premise_gwp100_inputdf = prem_gwp_dfraw, cf_inputds = cf_point, ssp = sp, fairMY = mmy, GWP100_only = True )
        minorg, allg = xx.get_minorand_allGHG()
        emt_C =  xx.prep_empty_C(allg)
        fullminor_C = xx.assign_minorghg_to_C_GWP100(emt_C, minorg )
        #print(fullminor_C.head() )
        full_allC = xx.assign_majorghg_dCC(fullminor_C)
        data = xx.prep_data_for_bw2method (full_allC, 
                                           mybio = bw2data.Database("ecoinvent-3.11-biosphere") , 
                                           mybio_str = "ecoinvent-3.11-biosphere")
        print(len(data))
        xx.prep_final_dCC_bw2method (C = full_allC, data = data, gwp_method = '- fixed-AGWPCO2')

start preparing data to be assigned as new bw2data.methods, for SSP 119 and fair_MY2030
2
start creating new method, under: SSP119, MY2030 
finishing preparing new methods, method name : ('Climate Change prospective GWP100', 'SSP119', 'MY2030', 'pGWP100 - fixed-AGWPCO2')
start preparing data to be assigned as new bw2data.methods, for SSP 126 and fair_MY2030
2
start creating new method, under: SSP126, MY2030 
finishing preparing new methods, method name : ('Climate Change prospective GWP100', 'SSP126', 'MY2030', 'pGWP100 - fixed-AGWPCO2')
start preparing data to be assigned as new bw2data.methods, for SSP 245 and fair_MY2030
2
start creating new method, under: SSP245, MY2030 
finishing preparing new methods, method name : ('Climate Change prospective GWP100', 'SSP245', 'MY2030', 'pGWP100 - fixed-AGWPCO2')
start preparing data to be assigned as new bw2data.methods, for SSP 434 and fair_MY2030
2
start creating new method, under: SSP434, MY2030 
finishing preparing new methods, method name

In [21]:
### testing use the new method, caz it cant be opened in AB3 
ei = bw2data.Database("ecoinvent-3.11-cutoff")
act = ei.random() 
print(act)

gwp_key = [
    m for m in bw2data.methods if "Climate Change prospective GWP100" in str(m)
].pop()
print(gwp_key)

my_functional_unit, data_objs, _ = bw2data.prepare_lca_inputs(
    {act: 1},
    method=gwp_key,
)

my_lca = bw2calc.LCA(demand=my_functional_unit, data_objs=data_objs)
my_lca.lci()
my_lca.lcia()
my_lca.score

'market for biomethane, medium pressure, vehicle grade' (kilogram, CH, None)
('Climate Change prospective GWP100', 'SSP119', 'MY2030', 'pGWP100 - fixed-AGWPCO2')


1.6841410196588589